In [1]:
# ============================================================
# EPL Transformer Trajectory Model (NO-ODDS) — SAFE VERSION
# - Learns team performance trajectories with a Transformer
# - Builds per-team sequences of last N games BEFORE each match
# - Adds context: current table points/rank, rest days, etc.
# - Time split + temperature scaling calibration
# ============================================================

import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -------------------- CONFIG --------------------
CSV_PATH    = "epl_master_train_full.csv"
MIN_DATE    = "2005-01-01"
NAN_THRESH  = 0.90
RANDOM_SEED = 42

SEQ_LEN     = 12          # how many past matches per team to feed transformer
BATCH_SIZE  = 256
EPOCHS      = 10
LR          = 2e-4
WEIGHT_DECAY= 1e-2

D_MODEL     = 96
N_HEAD      = 4
N_LAYERS    = 3
D_FF        = 256
DROPOUT     = 0.1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

fixtures = [
    ("2026-02-28", "Wolves",       "Aston Villa"),
    ("2026-02-28", "Bournemouth",  "Sunderland"),
    ("2026-02-28", "Newcastle",    "Everton"),
    ("2026-02-28", "Burnley",      "Brentford"),
    ("2026-02-28", "Liverpool",    "West Ham"),
    ("2026-02-28", "Leeds",        "Man City"),
]
fixtures_df = pd.DataFrame(fixtures, columns=["date","home_team","away_team"])
fixtures_df["date"] = pd.to_datetime(fixtures_df["date"])

# -------------------- helper: team name cleaning --------------------
def clean_team_name(x: str) -> str:
    if pd.isna(x):
        return x
    s = str(x).strip()
    repl = {
        "Wolverhampton": "Wolves",
        "Manchester City": "Man City",
        "Manchester United": "Man United",
        "West Ham United": "West Ham",
        "Nott'm Forest": "Nottm Forest",
        "Nottingham Forest": "Nottm Forest",
        "Spurs": "Tottenham",
    }
    return repl.get(s, s)

# -------------------- load + cleanup --------------------
df0 = pd.read_csv(CSV_PATH, low_memory=False)
df0["date"] = pd.to_datetime(df0["date"], errors="coerce")

rename_map = {
    "HomeTeam": "home_team", "AwayTeam": "away_team",
    "FTHG": "home_goals", "FTAG": "away_goals", "FTR": "ft_result"
}
for k, v in rename_map.items():
    if k in df0.columns and v not in df0.columns:
        df0 = df0.rename(columns={k: v})

df0 = df0.loc[:, df0.isna().mean() <= NAN_THRESH].copy()

need = ["date","home_team","away_team","home_goals","away_goals","target"]
missing = [c for c in need if c not in df0.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}\nFound: {list(df0.columns)[:30]}")

df0["home_team"] = df0["home_team"].map(clean_team_name)
df0["away_team"] = df0["away_team"].map(clean_team_name)

df0 = df0[df0["date"].notna() & (df0["date"] >= pd.to_datetime(MIN_DATE))].copy()
df0["target"] = df0["target"].astype(str).str.strip()
df0 = df0[df0["target"].isin(["H","D","A"])].copy()

df0["home_goals"] = pd.to_numeric(df0["home_goals"], errors="coerce")
df0["away_goals"] = pd.to_numeric(df0["away_goals"], errors="coerce")
df0 = df0[df0["home_goals"].notna() & df0["away_goals"].notna()].copy()

df0 = df0.sort_values("date").reset_index(drop=True)

# -------------------- points helper --------------------
def result_points(hg, ag):
    if hg > ag:  return 3, 0
    if hg < ag:  return 0, 3
    return 1, 1

# ============================================================
# 1) Build a "long" per-team history table (past games)
#    Each row = one team in one match, with stats from that match.
# ============================================================
rows = []
for i, r in df0.iterrows():
    hg, ag = float(r["home_goals"]), float(r["away_goals"])
    hp, ap = result_points(hg, ag)
    dt = r["date"]

    # home team view
    rows.append({
        "match_idx": i, "date": dt, "team": r["home_team"], "opp": r["away_team"],
        "is_home": 1,
        "gf": hg, "ga": ag, "gd": hg - ag, "pts": hp,
    })
    # away team view
    rows.append({
        "match_idx": i, "date": dt, "team": r["away_team"], "opp": r["home_team"],
        "is_home": 0,
        "gf": ag, "ga": hg, "gd": ag - hg, "pts": ap,
    })

long = pd.DataFrame(rows).sort_values(["date","match_idx","team"]).reset_index(drop=True)

# ============================================================
# 2) Compute "table" (rank/points) over time, strictly causal.
#    For each match_idx, compute team points BEFORE that match.
# ============================================================
teams = sorted(pd.unique(pd.concat([df0["home_team"], df0["away_team"]], ignore_index=True)))
team_to_id = {t:i for i,t in enumerate(teams)}

# state variables updated as we move through time
points = {t: 0.0 for t in teams}
gf_tot = {t: 0.0 for t in teams}
ga_tot = {t: 0.0 for t in teams}
last_date = {t: None for t in teams}

# We'll store per match_idx: home/away table context BEFORE updating with that match.
ctx_rows = []

for i, r in df0.iterrows():
    dt = r["date"]
    ht = r["home_team"]; at = r["away_team"]
    hg = float(r["home_goals"]); ag = float(r["away_goals"])
    hp, ap = result_points(hg, ag)

    # compute current ranks BEFORE updating this match
    # ranking key: points, goal diff, goals for
    table = []
    for t in teams:
        gd = gf_tot[t] - ga_tot[t]
        table.append((t, points[t], gd, gf_tot[t]))
    table_sorted = sorted(table, key=lambda x: (x[1], x[2], x[3]), reverse=True)
    rank_map = {t: (rk+1) for rk, (t,_,_,_) in enumerate(table_sorted)}

    # rest days BEFORE match
    def rest_days(team):
        if last_date[team] is None:
            return np.nan
        return float((dt - last_date[team]).days)

    ctx_rows.append({
        "match_idx": i,
        "home_pts_pre": points[ht],
        "away_pts_pre": points[at],
        "home_rank_pre": rank_map[ht],
        "away_rank_pre": rank_map[at],
        "home_rest_days": rest_days(ht),
        "away_rest_days": rest_days(at),
    })

    # update state AFTER match
    points[ht] += hp; points[at] += ap
    gf_tot[ht] += hg; ga_tot[ht] += ag
    gf_tot[at] += ag; ga_tot[at] += hg
    last_date[ht] = dt; last_date[at] = dt

ctx = pd.DataFrame(ctx_rows).set_index("match_idx")

# merge context into df
df = df0.join(ctx, how="left")

# normalize a few context signals (model will train better)
# (We do simple z-score with train stats later; here we keep raw.)

# ============================================================
# 3) Build sequences: for each match, pull last SEQ_LEN games
#    for the home team and away team, strictly BEFORE match_idx.
# ============================================================

# We will build for each team a list of its long rows (by date/match_idx),
# then for each match, extract last SEQ_LEN rows where match_idx < current.
long_by_team = {t: long[long["team"] == t].sort_values(["date","match_idx"]).reset_index(drop=True) for t in teams}

# What goes into each "token" (per past game)?
# Keep it small and numeric.
TOKEN_COLS = ["is_home", "gf", "ga", "gd", "pts"]

def get_team_seq(team: str, match_idx: int):
    hist = long_by_team[team]
    # strictly past games
    past = hist[hist["match_idx"] < match_idx]
    if len(past) < SEQ_LEN:
        return None  # not enough history
    tail = past.iloc[-SEQ_LEN:][TOKEN_COLS].to_numpy(dtype=np.float32)
    return tail  # shape (SEQ_LEN, token_dim)

def build_dataset_rows():
    rows = []
    for i, r in df.iterrows():
        ht, at = r["home_team"], r["away_team"]
        home_seq = get_team_seq(ht, i)
        away_seq = get_team_seq(at, i)
        if home_seq is None or away_seq is None:
            continue

        rows.append({
            "match_idx": i,
            "home_team": ht,
            "away_team": at,
            "home_seq": home_seq,
            "away_seq": away_seq,
            # context
            "home_pts_pre": float(r["home_pts_pre"]),
            "away_pts_pre": float(r["away_pts_pre"]),
            "home_rank_pre": float(r["home_rank_pre"]),
            "away_rank_pre": float(r["away_rank_pre"]),
            "home_rest_days": float(r["home_rest_days"]) if np.isfinite(r["home_rest_days"]) else np.nan,
            "away_rest_days": float(r["away_rest_days"]) if np.isfinite(r["away_rest_days"]) else np.nan,
            # target
            "target": r["target"],
        })
    out = pd.DataFrame(rows)
    return out

data = build_dataset_rows()
data = data.sort_values("match_idx").reset_index(drop=True)

# fill NaNs in rest days (season start) with median
for c in ["home_rest_days","away_rest_days"]:
    med = np.nanmedian(data[c].to_numpy())
    data[c] = data[c].fillna(med)

print("Matches with enough history:", len(data), "out of", len(df0))

# encode target
le = LabelEncoder()
y_enc = le.fit_transform(data["target"].values)
class_names = list(le.classes_)
print("Classes:", class_names)

# ============================================================
# 4) Time split
# ============================================================
split_idx = int(len(data) * 0.85)
train_df = data.iloc[:split_idx].copy()
test_df  = data.iloc[split_idx:].copy()

# ============================================================
# 5) Normalize numeric token features + context using train stats
# ============================================================
# tokens
token_stack = np.concatenate(train_df["home_seq"].values.tolist() + train_df["away_seq"].values.tolist(), axis=0)
tok_mean = token_stack.mean(axis=0)
tok_std  = token_stack.std(axis=0) + 1e-6

# context
CTX_COLS = ["home_pts_pre","away_pts_pre","home_rank_pre","away_rank_pre","home_rest_days","away_rest_days"]
ctx_mat = train_df[CTX_COLS].to_numpy(dtype=np.float32)
ctx_mean = ctx_mat.mean(axis=0)
ctx_std  = ctx_mat.std(axis=0) + 1e-6

def norm_tokens(x):
    return (x - tok_mean) / tok_std

def norm_ctx(x):
    return (x - ctx_mean) / ctx_std

# ============================================================
# 6) Dataset / Dataloader
# ============================================================
class EPLSeqDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, y: np.ndarray):
        self.f = frame.reset_index(drop=True)
        self.y = y.astype(np.int64)

    def __len__(self):
        return len(self.f)

    def __getitem__(self, idx):
        r = self.f.iloc[idx]
        home_seq = norm_tokens(r["home_seq"]).astype(np.float32)
        away_seq = norm_tokens(r["away_seq"]).astype(np.float32)

        ctx = norm_ctx(r[CTX_COLS].to_numpy(dtype=np.float32))

        # add team ids as embeddings (like subject id in EEG)
        home_id = team_to_id[r["home_team"]]
        away_id = team_to_id[r["away_team"]]

        return (
            torch.from_numpy(home_seq),            # (T, token_dim)
            torch.from_numpy(away_seq),            # (T, token_dim)
            torch.tensor(ctx, dtype=torch.float32),# (ctx_dim,)
            torch.tensor(home_id, dtype=torch.long),
            torch.tensor(away_id, dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.long),
        )

y_train = le.transform(train_df["target"].values)
y_test  = le.transform(test_df["target"].values)

train_ds = EPLSeqDataset(train_df, y_train)
test_ds  = EPLSeqDataset(test_df,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# ============================================================
# 7) Model: Transformer Encoder for home_seq and away_seq
# ============================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, T, D)
        T = x.size(1)
        return x + self.pe[:, :T, :]

class TrajTransformer(nn.Module):
    def __init__(self, token_dim, n_teams, ctx_dim, n_classes):
        super().__init__()

        self.team_emb = nn.Embedding(n_teams, D_MODEL)

        self.in_proj = nn.Linear(token_dim, D_MODEL)
        self.pos = PositionalEncoding(D_MODEL, max_len=SEQ_LEN)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=D_FF,
            dropout=DROPOUT, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=N_LAYERS)

        # pooling: attention-like learnable CLS token
        self.cls = nn.Parameter(torch.zeros(1, 1, D_MODEL))
        nn.init.normal_(self.cls, std=0.02)

        # context head
        self.ctx_mlp = nn.Sequential(
            nn.Linear(ctx_dim, D_MODEL),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(D_MODEL, D_MODEL),
        )

        # final classifier
        self.head = nn.Sequential(
            nn.Linear(D_MODEL*3, 256),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(256, n_classes)
        )

    def encode_team(self, seq, team_id):
        # seq: (B, T, token_dim)
        B, T, _ = seq.shape
        x = self.in_proj(seq)  # (B,T,D)
        x = self.pos(x)

        # add CLS
        cls = self.cls.expand(B, 1, -1)  # (B,1,D)
        x = torch.cat([cls, x], dim=1)   # (B,T+1,D)

        # add team embedding as a bias term (like subject embedding)
        te = self.team_emb(team_id).unsqueeze(1)  # (B,1,D)
        x = x + te

        z = self.encoder(x)  # (B,T+1,D)
        pooled = z[:, 0, :]  # CLS
        return pooled

    def forward(self, home_seq, away_seq, ctx, home_id, away_id):
        h = self.encode_team(home_seq, home_id)
        a = self.encode_team(away_seq, away_id)
        c = self.ctx_mlp(ctx)

        feat = torch.cat([h, a, c], dim=1)
        logits = self.head(feat)
        return logits

token_dim = len(TOKEN_COLS)
ctx_dim = len(CTX_COLS)
n_classes = len(class_names)

model = TrajTransformer(token_dim, n_teams=len(teams), ctx_dim=ctx_dim, n_classes=n_classes).to(DEVICE)

# ============================================================
# 8) Train
# ============================================================
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

criterion = nn.CrossEntropyLoss()
optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0.0
    all_logits = []
    all_y = []
    for home_seq, away_seq, ctx, home_id, away_id, y in loader:
        home_seq = home_seq.to(DEVICE)
        away_seq = away_seq.to(DEVICE)
        ctx = ctx.to(DEVICE)
        home_id = home_id.to(DEVICE)
        away_id = away_id.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(home_seq, away_seq, ctx, home_id, away_id)
        loss = criterion(logits, y)

        if train:
            optim.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

        total_loss += float(loss.item()) * y.size(0)
        all_logits.append(logits.detach().cpu())
        all_y.append(y.detach().cpu())

    all_logits = torch.cat(all_logits, dim=0).numpy()
    all_y = torch.cat(all_y, dim=0).numpy()
    probs = torch.softmax(torch.from_numpy(all_logits), dim=1).numpy()
    return total_loss / len(loader.dataset), log_loss(all_y, probs)

best_test = 1e9
for ep in range(1, EPOCHS+1):
    tr_loss, tr_ll = run_epoch(train_loader, train=True)
    te_loss, te_ll = run_epoch(test_loader, train=False)
    print(f"Epoch {ep:02d} | train loss {tr_loss:.4f} ll {tr_ll:.4f} | test loss {te_loss:.4f} ll {te_ll:.4f}")
    if te_ll < best_test:
        best_test = te_ll
        best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}

model.load_state_dict(best_state)
print("Best test log loss:", best_test)

# ============================================================
# 9) Temperature scaling calibration (simple + effective)
# ============================================================
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.logT = nn.Parameter(torch.zeros(()))

    def forward(self, logits):
        T = torch.exp(self.logT) + 1e-6
        return logits / T

scaler = TemperatureScaler().to(DEVICE)
optT = torch.optim.LBFGS(scaler.parameters(), lr=0.1, max_iter=50)

# collect test logits
model.eval()
test_logits = []
test_y = []
with torch.no_grad():
    for home_seq, away_seq, ctx, home_id, away_id, y in test_loader:
        logits = model(home_seq.to(DEVICE), away_seq.to(DEVICE), ctx.to(DEVICE), home_id.to(DEVICE), away_id.to(DEVICE))
        test_logits.append(logits)
        test_y.append(y.to(DEVICE))
test_logits = torch.cat(test_logits, dim=0)
test_y = torch.cat(test_y, dim=0)

def closure():
    optT.zero_grad()
    loss = nn.CrossEntropyLoss()(scaler(test_logits), test_y)
    loss.backward()
    return loss

optT.step(closure)
T = float(torch.exp(scaler.logT).detach().cpu())
print("Calibrated Temperature T:", T)

# ============================================================
# 10) Predict fixtures: build latest sequences + current table context
# ============================================================
# Build "latest seq lookup": last SEQ_LEN tokens per team from long
def latest_team_seq(team: str):
    team = clean_team_name(team)
    hist = long_by_team.get(team, None)
    if hist is None or len(hist) < SEQ_LEN:
        return None
    tail = hist.iloc[-SEQ_LEN:][TOKEN_COLS].to_numpy(dtype=np.float32)
    return tail

# Recompute table/ranks at end of history (for fixture context)
# We’ll use the final computed state from the ctx-building pass:
final_points = points
final_gf = gf_tot
final_ga = ga_tot

table = []
for t in teams:
    gd = final_gf[t] - final_ga[t]
    table.append((t, final_points[t], gd, final_gf[t]))
table_sorted = sorted(table, key=lambda x: (x[1], x[2], x[3]), reverse=True)
final_rank = {t: (rk+1) for rk, (t,_,_,_) in enumerate(table_sorted)}

# rest days for fixtures: days since last match in history
final_last_date = last_date

def fixture_ctx(dt, ht, at):
    ht = clean_team_name(ht); at = clean_team_name(at)
    def rest(team):
        ld = final_last_date.get(team, None)
        if ld is None:
            return np.nan
        return float((pd.to_datetime(dt) - ld).days)

    c = np.array([
        final_points.get(ht, 0.0),
        final_points.get(at, 0.0),
        float(final_rank.get(ht, len(teams))),
        float(final_rank.get(at, len(teams))),
        rest(ht),
        rest(at)
    ], dtype=np.float32)

    # fill nan rest with median (like train)
    c[np.isnan(c)] = np.nanmedian(ctx_mat[:, [4,5]])
    return c

def predict_fixture(dt, ht, at):
    hs = latest_team_seq(ht)
    asq = latest_team_seq(at)
    if hs is None or asq is None:
        return None

    c = fixture_ctx(dt, ht, at)

    hs_t = torch.from_numpy(norm_tokens(hs)).unsqueeze(0).to(DEVICE)   # (1,T,dim)
    as_t = torch.from_numpy(norm_tokens(asq)).unsqueeze(0).to(DEVICE)
    c_t  = torch.from_numpy(norm_ctx(c)).unsqueeze(0).to(DEVICE)

    hid = torch.tensor([team_to_id[clean_team_name(ht)]], dtype=torch.long).to(DEVICE)
    aid = torch.tensor([team_to_id[clean_team_name(at)]], dtype=torch.long).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(hs_t, as_t, c_t, hid, aid)
        logits = scaler(logits)  # calibrated
        p = torch.softmax(logits, dim=1).cpu().numpy()[0]

    out = dict(zip(class_names, p.tolist()))
    return out

# run fixtures
rows_out = []
for _, r in fixtures_df.iterrows():
    dt, ht, at = r["date"], r["home_team"], r["away_team"]
    p = predict_fixture(dt, ht, at)
    if p is None:
        rows_out.append([dt, ht, at, np.nan, np.nan, np.nan, "NO DATA", np.nan])
        continue
    pH = p.get("H", np.nan); pD = p.get("D", np.nan); pA = p.get("A", np.nan)
    pick = max([("home",pH),("draw",pD),("away",pA)], key=lambda x: x[1])[0]
    conf = max(pH, pD, pA)
    rows_out.append([dt, ht, at, pH, pD, pA, pick, conf])

summary = pd.DataFrame(rows_out, columns=["date","home_team","away_team","p_home","p_draw","p_away","pick","confidence"])
print("\n================= NEXT WEEK SUMMARY (Transformer) =================")
display(summary.sort_values("confidence", ascending=False).reset_index(drop=True))

def print_match_table(row):
    dt = pd.to_datetime(row["date"]).date()
    ht = row["home_team"]; at = row["away_team"]
    print(f"\n{dt} | {ht} vs {at}")
    if pd.isna(row["p_home"]):
        print("⚠️ Not enough team history for SEQ_LEN =", SEQ_LEN)
        return
    print(f"Suggestion: {str(row['pick']).upper()} (confidence={row['confidence']:.3f})")
    table = pd.DataFrame({
        "Outcome": ["Home (H)","Draw (D)","Away (A)"],
        "p_model": [row["p_home"], row["p_draw"], row["p_away"]],
    })
    display(table.style.format({"p_model":"{:.3f}"}))

for _, rr in summary.sort_values("date").iterrows():
    print_match_table(rr)

print("\n✅ Done.")

/cbica/home/dadashkj/.conda/envs/pushkar/lib/python3.10/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


Matches with enough history: 7463 out of 7871
Classes: ['A', 'D', 'H']
Epoch 01 | train loss 1.0288 ll 1.0288 | test loss 1.0220 ll 1.0220
Epoch 02 | train loss 0.9744 ll 0.9744 | test loss 1.0241 ll 1.0241
Epoch 03 | train loss 0.9665 ll 0.9665 | test loss 1.0187 ll 1.0187
Epoch 04 | train loss 0.9618 ll 0.9618 | test loss 1.0193 ll 1.0193
Epoch 05 | train loss 0.9603 ll 0.9603 | test loss 1.0199 ll 1.0199
Epoch 06 | train loss 0.9578 ll 0.9578 | test loss 1.0147 ll 1.0147
Epoch 07 | train loss 0.9529 ll 0.9529 | test loss 1.0155 ll 1.0155
Epoch 08 | train loss 0.9507 ll 0.9507 | test loss 1.0150 ll 1.0150
Epoch 09 | train loss 0.9475 ll 0.9475 | test loss 1.0147 ll 1.0147
Epoch 10 | train loss 0.9478 ll 0.9478 | test loss 1.0156 ll 1.0156
Best test log loss: 1.0146849238151832
Calibrated Temperature T: 1.1759495735168457

================= NEXT WEEK SUMMARY (Transformer) =================


,date,home_team,away_team,p_home,p_draw,p_away,pick,confidence
0,2026-02-28,Liverpool,West Ham,0.642488,0.236836,0.120676,home,0.642488
1,2026-02-28,Leeds,Man City,0.213807,0.195811,0.590383,away,0.590383
2,2026-02-28,Wolves,Aston Villa,0.302107,0.233669,0.464224,away,0.464224
3,2026-02-28,Bournemouth,Sunderland,0.459675,0.314258,0.226068,home,0.459675
4,2026-02-28,Burnley,Brentford,0.268646,0.343979,0.387375,away,0.387375
5,2026-02-28,Newcastle,Everton,0.385155,0.311563,0.303282,home,0.385155



2026-02-28 | Wolves vs Aston Villa
Suggestion: AWAY (confidence=0.464)


,Outcome,p_model
0,Home (H),0.302
1,Draw (D),0.234
2,Away (A),0.464


...


2026-02-28 | Liverpool vs West Ham
Suggestion: HOME (confidence=0.642)


,Outcome,p_model
0,Home (H),0.642
1,Draw (D),0.237
2,Away (A),0.121



2026-02-28 | Leeds vs Man City
Suggestion: AWAY (confidence=0.590)


,Outcome,p_model
0,Home (H),0.214
1,Draw (D),0.196
2,Away (A),0.590



✅ Done.


In [3]:
# # ============================================================
# # SAVE: model + calibrator + label encoder + feature list
# # ============================================================

# import joblib

# MODEL_CBM_PATH = "epl_catboost.cbm"
# BUNDLE_PATH    = "epl_bundle.joblib"

# # 1) Save raw CatBoost model (fast + portable)
# model.save_model(MODEL_CBM_PATH)

# # 2) Save calibrated wrapper + metadata needed for inference
# bundle = {
#     "calibrator": cal,          # CalibratedClassifierCV wrapping model
#     "label_encoder": le,        # LabelEncoder with classes_ mapping
#     "feature_cols": feature_cols,
#     "cat_feats": cat_feats,
#     "use_odds": USE_ODDS,
#     "min_date": MIN_DATE,
#     "nan_thresh": NAN_THRESH,
# }

# joblib.dump(bundle, BUNDLE_PATH)

# print("✅ Saved:")
# print(" - CatBoost model:", MODEL_CBM_PATH)
# print(" - Full bundle   :", BUNDLE_PATH)
# print("Label classes:", list(le.classes_))
# print("Features:", len(feature_cols))

In [7]:
# ============================================================
# Interactive "Next Game" UI
# ✅ If ANY odds cell is empty -> fill missing odds from HISTORICAL odds in df0
# ✅ Accepts American (+330/-120) OR Decimal (4.30/1.83) in the boxes
# ✅ If no exact fixture history, falls back to team-level (home team at home, away team away)
# ✅ If still none, falls back to league median odds (last resort)
# ============================================================

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---- requires these to exist from training cell ----
# df0, cal, le, feature_cols, ROLL_N, clean_team_name, get_latest_rolls

# -------------------- pick which odds columns exist in df0 --------------------
# We'll try common names. Use the first triplet we find.
ODDS_CANDIDATES = [
    ("odds_home","odds_draw","odds_away"),
    ("B365H","B365D","B365A"),
    ("PSH","PSD","PSA"),
    ("WHH","WHD","WHA"),
    ("VCH","VCD","VCA"),
]

def _find_odds_cols(df):
    for h,d,a in ODDS_CANDIDATES:
        if {h,d,a}.issubset(df.columns):
            return (h,d,a)
    return None

odds_cols = _find_odds_cols(df0)
if odds_cols is None:
    print("⚠️ No odds columns found in df0. Historical-odds fallback will NOT work.")
    print("   Add odds columns (e.g., odds_home/odds_draw/odds_away or B365H/B365D/B365A) to df0, or enter odds manually.")
else:
    print(f"✅ Using historical odds columns: {odds_cols}")

# -------------------- helpers --------------------
class_names = list(le.classes_)
def idx(label): return class_names.index(label)

def _is_finite(x):
    return x is not None and np.isfinite(x)

def _parse_user_odds_to_decimal(s: str):
    """
    Accepts:
      - "" -> None
      - "+330" / "-120" (American) -> decimal
      - "4.30" / "1.83" (Decimal) -> decimal
      - "330" (assume American) -> decimal
    """
    s = (s or "").strip()
    if s == "":
        return None

    # normalize
    s2 = s.replace(" ", "")
    try:
        # If user typed "+330" or "-120"
        if s2.startswith("+") or s2.startswith("-"):
            a = float(s2)
            # American to decimal
            if a > 0:
                return 1.0 + (a / 100.0)
            else:
                return 1.0 + (100.0 / abs(a))

        # else numeric
        x = float(s2)

        # Heuristic:
        # - Decimal odds are typically > 1.01 and often < 20
        # - American odds are often >= 100 in magnitude
        if x >= 1.01 and x <= 25.0:
            # treat as decimal
            return x
        else:
            # treat as American (e.g., 330, 120, -120)
            a = x
            if a > 0:
                return 1.0 + (a / 100.0)
            else:
                return 1.0 + (100.0 / abs(a))
    except Exception:
        return None

def implied_probs_from_decimal_odds(dec_h, dec_d, dec_a):
    """
    Convert decimal odds -> (p_fair_home, p_fair_draw, p_fair_away, overround).
    """
    if not (_is_finite(dec_h) and _is_finite(dec_d) and _is_finite(dec_a)):
        return None, None, None, None

    p_raw = np.array([1/dec_h, 1/dec_d, 1/dec_a], dtype=float)
    overround = float(p_raw.sum() - 1.0)
    p_fair = p_raw / p_raw.sum()
    return float(p_fair[0]), float(p_fair[1]), float(p_fair[2]), float(overround)

def get_historical_odds_decimal(home_team, away_team, n_last=8):
    """
    Returns (dec_h, dec_d, dec_a, source_str) using df0.
    Priority:
      1) Exact fixture history: same home_team & away_team (last n_last)
      2) Team-level: home team at home (any opponent) + away team away (any opponent) (last n_last each)
         - combine by median
      3) League fallback: median odds across all matches (last resort)
    """
    if odds_cols is None:
        return None, None, None, "no_odds_columns"

    hcol, dcol, acol = odds_cols
    ht = clean_team_name(home_team)
    at = clean_team_name(away_team)

    dfx = df0.copy()
    # ensure cleaned teams (in case df0 isn’t already cleaned)
    dfx["home_team"] = dfx["home_team"].map(clean_team_name)
    dfx["away_team"] = dfx["away_team"].map(clean_team_name)
    dfx = dfx.sort_values("date")

    def _med_triplet(sub):
        sub = sub[[hcol, dcol, acol]].apply(pd.to_numeric, errors="coerce")
        sub = sub.dropna()
        if len(sub) == 0:
            return None
        m = sub.median()
        return float(m[hcol]), float(m[dcol]), float(m[acol])

    # 1) exact fixture
    exact = dfx[(dfx["home_team"] == ht) & (dfx["away_team"] == at)].tail(n_last)
    t = _med_triplet(exact)
    if t is not None:
        return (*t, f"exact_fixture_median_last{min(n_last, len(exact))}")

    # 2) team-level: home team at home + away team away
    home_hist = dfx[dfx["home_team"] == ht].tail(n_last)
    away_hist = dfx[dfx["away_team"] == at].tail(n_last)

    t1 = _med_triplet(home_hist)
    t2 = _med_triplet(away_hist)

    if t1 is not None or t2 is not None:
        vals = []
        if t1 is not None: vals.append(t1)
        if t2 is not None: vals.append(t2)
        dec_h = float(np.median([v[0] for v in vals]))
        dec_d = float(np.median([v[1] for v in vals]))
        dec_a = float(np.median([v[2] for v in vals]))
        return dec_h, dec_d, dec_a, "team_level_median"

    # 3) league fallback
    league = dfx.tail(4000)  # keep it reasonably recent-ish; tweak if you want
    t3 = _med_triplet(league)
    if t3 is not None:
        return (*t3, "league_median")

    return None, None, None, "no_history_found"

def build_fixture_row(date_val, home_team, away_team):
    fx = pd.DataFrame([{
        "date": pd.to_datetime(date_val),
        "home_team": clean_team_name(home_team),
        "away_team": clean_team_name(away_team),
    }])

    fx[[f"home_roll{ROLL_N}_pts_calc", f"home_roll{ROLL_N}_gf_calc", f"home_roll{ROLL_N}_ga_calc"]] = fx["home_team"].apply(
        lambda t: pd.Series(get_latest_rolls(t))
    )
    fx[[f"away_roll{ROLL_N}_pts_calc", f"away_roll{ROLL_N}_gf_calc", f"away_roll{ROLL_N}_ga_calc"]] = fx["away_team"].apply(
        lambda t: pd.Series(get_latest_rolls(t))
    )

    fx[f"pts_diff_{ROLL_N}_calc"] = fx[f"home_roll{ROLL_N}_pts_calc"] - fx[f"away_roll{ROLL_N}_pts_calc"]
    fx[f"gf_diff_{ROLL_N}_calc"]  = fx[f"home_roll{ROLL_N}_gf_calc"]  - fx[f"away_roll{ROLL_N}_gf_calc"]
    fx[f"ga_diff_{ROLL_N}_calc"]  = fx[f"home_roll{ROLL_N}_ga_calc"]  - fx[f"away_roll{ROLL_N}_ga_calc"]
    return fx

def predict_one(fx_row):
    P = cal.predict_proba(fx_row[feature_cols])
    pH = P[0, idx("H")] if "H" in class_names else np.nan
    pD = P[0, idx("D")] if "D" in class_names else np.nan
    pA = P[0, idx("A")] if "A" in class_names else np.nan
    return float(pH), float(pD), float(pA)

# -------------------- UI widgets --------------------
teams = sorted(pd.unique(pd.concat([df0["home_team"], df0["away_team"]], ignore_index=True)).tolist())

date_picker = widgets.DatePicker(description="Date:", value=pd.Timestamp("2026-02-28").date())

home_dd = widgets.Dropdown(options=teams, description="Home:", value="Wolves" if "Wolves" in teams else teams[0],
                           layout=widgets.Layout(width="320px"))
away_dd = widgets.Dropdown(options=teams, description="Away:", value="Aston Villa" if "Aston Villa" in teams else teams[1],
                           layout=widgets.Layout(width="320px"))

# odds inputs (leave blank to trigger historical fill)
odds_home = widgets.Text(description="Odds H:", value="", placeholder="e.g. +330 or 4.30", layout=widgets.Layout(width="260px"))
odds_draw = widgets.Text(description="Odds D:", value="", placeholder="e.g. +260 or 3.60", layout=widgets.Layout(width="260px"))
odds_away = widgets.Text(description="Odds A:", value="", placeholder="e.g. -120 or 1.83", layout=widgets.Layout(width="260px"))

edge_thresh = widgets.FloatSlider(
    description="Edge ≥", min=0.0, max=0.15, step=0.005, value=0.03,
    readout_format=".3f", layout=widgets.Layout(width="420px")
)

btn = widgets.Button(description="Predict", button_style="primary", icon="check")
out = widgets.Output()

# -------------------- click logic --------------------
def on_click(_):
    with out:
        clear_output(wait=True)

        dt = date_picker.value
        ht = home_dd.value
        at = away_dd.value

        if ht == at:
            print("⚠️ Home and Away can’t be the same team.")
            return

        # ---- model features ----
        fx = build_fixture_row(dt, ht, at)
        if fx[feature_cols].isna().any(axis=1).iloc[0]:
            print("⚠️ Missing rolling history for one of these teams (not in data / not enough history).")
            display(fx[["date","home_team","away_team"]])
            return

        pH, pD, pA = predict_one(fx)
        probs = pd.Series({"H": pH, "D": pD, "A": pA})
        pick = probs.idxmax()
        conf = probs.max()

        # ---- odds: user -> decimal; fill missing from history ----
        user_dec_h = _parse_user_odds_to_decimal(odds_home.value)
        user_dec_d = _parse_user_odds_to_decimal(odds_draw.value)
        user_dec_a = _parse_user_odds_to_decimal(odds_away.value)

        need_fill = (user_dec_h is None) or (user_dec_d is None) or (user_dec_a is None)

        hist_dec_h = hist_dec_d = hist_dec_a = None
        hist_src = None
        if need_fill:
            hist_dec_h, hist_dec_d, hist_dec_a, hist_src = get_historical_odds_decimal(ht, at)

        dec_h = user_dec_h if user_dec_h is not None else hist_dec_h
        dec_d = user_dec_d if user_dec_d is not None else hist_dec_d
        dec_a = user_dec_a if user_dec_a is not None else hist_dec_a

        # If still missing, we can only show model probs
        have_all_odds = _is_finite(dec_h) and _is_finite(dec_d) and _is_finite(dec_a)

        # ---- header ----
        print(f"{pd.to_datetime(dt).date()} | {clean_team_name(ht)} vs {clean_team_name(at)}")
        print(f"Model pick: {pick}  (confidence={conf:.3f})")

        if have_all_odds:
            pbh, pbd, pba, overround = implied_probs_from_decimal_odds(dec_h, dec_d, dec_a)
            if hist_src is not None and need_fill:
                print(f"Odds filled from history: {hist_src}")
            print(f"Odds used (decimal): H={dec_h:.3f}  D={dec_d:.3f}  A={dec_a:.3f}")
            print(f"Overround (book margin): {overround:.3f}")

            rows = []
            for lab, name, dec in [("H","Home (H)",dec_h), ("D","Draw (D)",dec_d), ("A","Away (A)",dec_a)]:
                p_model = float(probs[lab])
                p_book  = {"H": pbh, "D": pbd, "A": pba}[lab]
                edge    = p_model - p_book
                ev      = (p_model * dec) - 1.0
                rows.append([name, p_model, p_book, edge, ev])

            table = pd.DataFrame(rows, columns=["Outcome","p_model","p_book","edge","EV"])

            best = table.loc[table["EV"].idxmax()]
            do_bet = (best["EV"] > 0) and (best["edge"] >= float(edge_thresh.value))

            print(f"Best EV side: {best['Outcome']} | EV={best['EV']:.3f} | edge={best['edge']:.3f}")
            print("Suggestion:", "BET ✅" if do_bet else "NO BET ❌")

            display(table.style.format({"p_model":"{:.3f}","p_book":"{:.3f}","edge":"{:.3f}","EV":"{:.3f}"}))
        else:
            print("Odds not provided (or could not be filled from history) → showing model probabilities only.")
            table = pd.DataFrame(
                [["Home (H)", float(probs["H"])],
                 ["Draw (D)", float(probs["D"])],
                 ["Away (A)", float(probs["A"])]],
                columns=["Outcome","p_model"]
            )
            display(table.style.format({"p_model":"{:.3f}"}))

btn.on_click(on_click)

ui1 = widgets.HBox([date_picker, home_dd, away_dd])
ui2 = widgets.HBox([odds_home, odds_draw, odds_away])
ui3 = widgets.HBox([edge_thresh, btn])

display(ui1, ui2, ui3, out)
print("✅ Leave any odds box blank to auto-fill from historical odds (if df0 has odds columns). Then click Predict.")

✅ Using historical odds columns: ('odds_home', 'odds_draw', 'odds_away')


Output()

✅ Leave any odds box blank to auto-fill from historical odds (if df0 has odds columns). Then click Predict.


In [4]:
import ipywidgets as widgets
from IPython.display import display
display(widgets.Dropdown(options=["works","nope"], description="Test:"))

Dropdown(description='Test:', options=('works', 'nope'), value='works')